# STL-10 with EmbeddingGemma 2

STL-10 has **13,000 labelled images**; 5,000 train (500 per class) and 8,000 test (800 per
class) - plus 100,000 unlabelled ones, all 96 x 96 colour.

This notebook embeds the **same 1,000 images** that `week_2` and `week_4` used, so the
vectors here can be compared directly with the ViT and autoencoder features.

Output: `data/embeddinggemma_stl10_1000.npz`.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from sentence_transformers import SentenceTransformer

## Load the model

`bfloat16` or `float32`

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

model = SentenceTransformer("google/embeddinggemma-2", model_kwargs={"torch_dtype": dtype})

print(dtype, "|", model.get_embedding_dimension())

## The same 1,000 images as before

`week_2` drew 1,000 images at random from the 13,000 labelled ones with seed 0, and every
later notebook and assignment reused those indices. Drawing them the same way reproduces
exactly the same sample, so the vectors stay comparable with the ViT and autoencoder
features.

The images come from `data/stl10_binary`, produced by `01_Download.ipynb`.

In [ ]:
DATA = Path("data/stl10_binary")
N, SEED = 1000, 0

class_names = [l.strip() for l in open(f"{DATA}/class_names.txt") if l.strip()]

X = np.concatenate([
    np.fromfile(f"{DATA}/train_X.bin", dtype=np.uint8).reshape(-1, 3, 96, 96),
    np.fromfile(f"{DATA}/test_X.bin", dtype=np.uint8).reshape(-1, 3, 96, 96),
]).transpose(0, 3, 2, 1)

y_all = np.concatenate([
    np.fromfile(f"{DATA}/train_y.bin", dtype=np.uint8),
    np.fromfile(f"{DATA}/test_y.bin", dtype=np.uint8),
])

idx = np.sort(np.random.default_rng(SEED).choice(len(X), N, replace=False))
y = y_all[idx]

imgs = [Image.fromarray(im) for im in X[idx]]

print(len(X), "labelled images ->", len(imgs), "sampled |", len(class_names), "classes")

fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))

for ax, k in zip(axes, range(6)):
    ax.imshow(imgs[k])
    ax.set_title(class_names[y[k] - 1], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Embed

Output is `data/embeddinggemma_stl10_1000.npz`, skipped if it already exists.

In [ ]:
EMB = Path("data/embeddinggemma_stl10_1000.npz")

if EMB.exists():
    d = np.load(EMB)
    E = d["E"]
    print(f"loaded {EMB} - skipped")
else:
    import time

    t0 = time.perf_counter()
    E = model.encode([{"image": im} for im in imgs], batch_size=16,
                     normalize_embeddings=True)
    dt = time.perf_counter() - t0

    EMB.parent.mkdir(exist_ok=True)
    np.savez_compressed(EMB, idx=idx, y=y, E=E.astype(np.float32))
    print(f"saved {EMB}  ({dt:.1f}s for {len(imgs)} images)")

print(E.shape, E.dtype)

## Hierarchical clustering

STL-10 has 10 classes. Cut the tree at **10** clusters and score the result with the
Adjusted Rand Index; 1.0 would mean the clusters and the classes agree perfectly, 0.0 is
what you would get from random labels.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score
import pandas as pd

D = np.clip(1 - E @ E.T, 0, 2)
np.fill_diagonal(D, 0.0)

Z = linkage(squareform(D, checks=False), method="complete")

clusters = fcluster(Z, t=10, criterion="maxclust")

print(f"clusters: {len(np.unique(clusters))}")
print(f"ARI     : {adjusted_rand_score(y, clusters):.3f}")

## Clusters are mixed

`y` holds the labels 1 to 10

In [ ]:
tab = pd.crosstab(clusters, y)
tab["sum"] = tab.sum(axis=1)

print(tab.to_string())

purity = tab.iloc[:, :10].max(axis=1) / tab["sum"]
print(f"\nclusters that are >=90% one class: {int((purity >= 0.9).sum())} of 10")

## How many features?

Two ways to give the clustering fewer numbers, swept over the same grid:

- **raw** - keep the first `n` entries of each vector
- **PCA** - project onto the first `n` principal components

Both are renormalised before the cosine distance, and both are cut at 10 clusters.

In [ ]:
from sklearn.decomposition import PCA


def cluster_ari(V, k=10):
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    D = np.clip(1 - V @ V.T, 0, 2)
    np.fill_diagonal(D, 0.0)
    Z = linkage(squareform(D, checks=False), method="complete")
    return adjusted_rand_score(y, fcluster(Z, t=k, criterion="maxclust"))


steps = range(32, 769, 32)

raw, pca = [], []

print(f"{'features':>8}  {'raw':>7}  {'PCA':>7}")
for n in steps:
    a = cluster_ari(E[:, :n])
    b = cluster_ari(PCA(n_components=n, random_state=0).fit_transform(E))
    raw.append(a)
    pca.append(b)
    print(f"{n:8}  {a:7.3f}  {b:7.3f}")

print(f"\nbest raw: {steps[int(np.argmax(raw))]} features -> {max(raw):.3f}")
print(f"best PCA: {steps[int(np.argmax(pca))]} components -> {max(pca):.3f}")

## The two curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(steps, raw, marker=".", label="first n features")
ax.plot(steps, pca, marker=".", label="PCA, n components")
ax.set_xlabel("number of features")
ax.set_ylabel("ARI")
ax.legend()

plt.tight_layout()
plt.show()